# Ejecuta tu primer circuito en hardware

*Primeros pasos con Qiskit y el hardware de IBM Quantum en este ejemplo de "Hola, mundo".*

<div class="alert alert-block alert-info">
<b>Versiones de paquetes</b>

El código de esta página se desarrolló con los siguientes requisitos. Recomendamos usar estas versiones o más recientes.

```
qiskit[all]~=2.5.2
qiskit-ibm-runtime~=0.47.0
```

*Nota de la traducción:* las celdas usan `qiskit_ibm_runtime.executor_estimator`, que existe a partir de **qiskit-ibm-runtime 0.50.0**. Instala esa versión o una más reciente.
</div>

Este ejemplo tiene dos partes. Primero crearás un programa cuántico sencillo de tipo "Hola, mundo" y lo ejecutarás en una unidad de procesamiento cuántico (QPU). Como la investigación cuántica real requiere programas mucho más robustos, en la segunda parte ([Escalar a un gran número de qubits](#Escalar-a-un-gran-número-de-qubits)) llevarás ese programa sencillo a escala de utilidad.

## 


          Instalar y autenticar

1. Si todavía no has instalado Qiskit, encontrarás las instrucciones en la guía [Inicio rápido









https://quantum.cloud.ibm.com/docs/es/guides/quick-start).

   * Instala el cliente de IBM Quantum Compute para ejecutar trabajos en hardware cuántico:

     ```bash
     pip install qiskit-ibm-runtime
     ```

   * Prepara un entorno para ejecutar notebooks de Jupyter de forma local:

     ```bash
     pip install jupyter
     ```

2. Configura tu autenticación para acceder al hardware cuántico con el [plan Open](https://quantum.cloud.ibm.com/docs/es/guides/plans-overview#open-plan), que es gratuito.

   (Si recibiste por correo una invitación para unirte a una cuenta, sigue en su lugar los [pasos para usuarios invitados](https://quantum.cloud.ibm.com/docs/es/guides/cloud-setup-invited).)

   * Ve a [IBM Quantum Platform](https://quantum.cloud.ibm.com/) para iniciar sesión o crear una cuenta.

     <div class="alert alert-block alert-info">
<b>Importante</b>

Si te conectas a través de un servidor proxy, debes usar `qiskit-ibm-runtime` v0.44.0 o posterior.
</div>

   * Genera tu clave de API (*API key*, también llamada *API token*) en el [panel principal](https://quantum.cloud.ibm.com/) y cópiala en un lugar seguro.

   * Ve a la página [Instances](https://quantum.cloud.ibm.com/instances) y busca la instancia que quieres usar. Pasa el cursor sobre su CRN y haz clic para copiarlo.

   * Guarda tus credenciales localmente con este código:

     ```python
     from qiskit_ibm_runtime import QiskitRuntimeService

     QiskitRuntimeService.save_account(
         # En `token`, usa la API_KEY de 44 caracteres que creaste
         # y guardaste desde el panel principal de IBM Quantum Platform
         token="<tu-api-key>",
         instance="<CRN>", # Opcional
     )
     ```

3. A partir de ahora puedes usar este código de Python cada vez que quieras autenticarte en el servicio IBM Quantum Compute:

   ```python
   from qiskit_ibm_runtime import QiskitRuntimeService

   # Ejecútalo cada vez que necesites el servicio
   service = QiskitRuntimeService()
   ```

<div class="alert alert-block alert-info">
<b>¿No estás en un entorno de Python de confianza?</b>

Si usas un computador público u otro entorno no seguro, sigue en su lugar las [instrucciones de autenticación manual](https://quantum.cloud.ibm.com/docs/es/guides/cloud-setup-untrusted) para mantener seguras tus credenciales.
</div>

## Crear y ejecutar un programa cuántico sencillo

Los cuatro pasos para escribir un programa cuántico con los patrones de Qiskit (*Qiskit patterns*) son:

1. Mapear el problema a un formato nativo cuántico.

2. Optimizar los circuitos y los operadores.

3. Ejecutar con una función primitiva cuántica.

4. Analizar los resultados.

### Paso 1. Mapear el problema a un formato nativo cuántico

En un programa cuántico, los *circuitos cuánticos* son el formato nativo para representar instrucciones cuánticas, y los *operadores* representan los observables que se van a medir. Para crear un circuito, normalmente se crea un objeto [`QuantumCircuit`](https://quantum.cloud.ibm.com/docs/es/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class) nuevo y luego se le agregan instrucciones en secuencia.

La siguiente celda de código crea un circuito que produce un *estado de Bell*, que es un estado en el que dos qubits están completamente entrelazados entre sí.

<div class="alert alert-block alert-info">
<b>Nota: orden de los bits</b>

El SDK de Qiskit numera los bits empezando por el menos significativo (*LSb 0*): el dígito $n$ vale $1 \ll n$, es decir, $2^n$. Para más detalles, consulta el tema [Orden de los bits en el SDK de Qiskit](https://quantum.cloud.ibm.com/docs/es/guides/bit-ordering).
</div>

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from matplotlib import pyplot as plt

# Primitiva antigua (del lado del servidor), anterior a qiskit-ibm-runtime 0.50.0
# from qiskit_ibm_runtime import EstimatorOptions
# from qiskit_ibm_runtime import Estimator

# Primitiva del lado del cliente, introducida en qiskit-ibm-runtime 0.50.0
from qiskit_ibm_runtime.options_models import EstimatorOptions
from qiskit_ibm_runtime.executor_estimator import Estimator

# Descomenta la siguiente línea si quieres usar un simulador:
# from qiskit_ibm_runtime.fake_provider import FakeBelemV2


# Crea un circuito nuevo con dos qubits
qc = QuantumCircuit(2)

# Agrega una compuerta Hadamard al qubit 0
qc.h(0)

# Aplica una compuerta X controlada al qubit 1, controlada por el qubit 0
qc.cx(0, 1)

# Devuelve un dibujo del circuito usando MatPlotLib ("mpl").
# Estas guías se escriben con notebooks de Jupyter, que
# muestran la salida de la última línea de cada celda.
# Si ejecutas esto en un script, usa `print(qc.draw())` para
# imprimir el dibujo en texto.
qc.draw("mpl")

Consulta [`QuantumCircuit`](https://quantum.cloud.ibm.com/docs/es/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class) en la documentación para ver todas las operaciones disponibles.

Al crear circuitos cuánticos también debes pensar qué tipo de datos quieres obtener después de la ejecución. Qiskit ofrece dos formas de obtener datos: puedes obtener muestras de las mediciones de los qubits que elijas medir, o puedes obtener el valor esperado de un observable. Prepara tu carga de trabajo para medir el circuito de una de estas dos formas con las primitivas (se explican en detalle en el [Paso 3](#Paso-3.-Ejecutar-con-las-primitivas-cuánticas)).

Este ejemplo mide valores esperados usando el submódulo `qiskit.quantum_info`, en el que se especifican operadores (objetos matemáticos que representan una acción o un proceso que cambia un estado cuántico). La siguiente celda de código crea seis operadores de Pauli de dos qubits: `IZ`, `IX`, `ZI`, `XI`, `ZZ` y `XX`.

In [ ]:
# Define seis observables distintos.

observables_labels = ["IZ", "IX", "ZI", "XI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]

<div class="alert alert-block alert-info">
<b>Notación de operadores</b>

Aquí, un operador como `ZZ` es una abreviatura del producto tensorial $Z\otimes Z$, que significa medir Z en el qubit 1 y Z en el qubit 0 a la vez, y obtener información sobre la correlación entre el qubit 1 y el qubit 0. Valores esperados como este se suelen escribir también como $\langle Z_1 Z_0 \rangle$.

Si el estado está entrelazado, la medición de $\langle Z_1 Z_0 \rangle$ debería ser distinta de la medición de $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$. Para el estado entrelazado concreto que crea el circuito anterior, la medición de $\langle Z_1 Z_0 \rangle$ debería ser 1 y la de $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$ debería ser cero.
</div>

### Paso 2. Optimizar los circuitos y los operadores

Al ejecutar circuitos en un dispositivo, es importante optimizar el conjunto de instrucciones del circuito y minimizar su profundidad total (aproximadamente, el número de instrucciones). Así se obtienen los mejores resultados posibles, porque se reducen los efectos del error y del ruido. Además, las instrucciones del circuito deben ajustarse a la [arquitectura del conjunto de instrucciones (ISA)](https://quantum.cloud.ibm.com/docs/es/guides/transpile#instruction-set-architecture) del dispositivo (*backend*) y tener en cuenta sus compuertas base y la conectividad entre sus qubits.

El siguiente código elige un dispositivo real al cual enviar un trabajo y transforma el circuito y los observables para que se ajusten a la ISA de ese dispositivo. Requiere que ya hayas [guardado tus credenciales](https://quantum.cloud.ibm.com/docs/es/guides/cloud-setup).

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(simulator=False, operational=True)

# Convierte a un circuito ISA y a observables mapeados al layout.
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

isa_circuit.draw("mpl", idle_wires=False)

### Paso 3. Ejecutar con las primitivas cuánticas

Los computadores cuánticos pueden producir resultados aleatorios, así que normalmente se recoge una muestra de los resultados ejecutando el circuito muchas veces. Puedes estimar el valor del observable con la clase `Estimator`. `Estimator` es una de las dos [primitivas](https://quantum.cloud.ibm.com/docs/es/guides/get-started-with-estimator); la otra es `Sampler`, que sirve para obtener datos de un computador cuántico. Estos objetos tienen un método `run()` que ejecuta la selección de circuitos, observables y parámetros (si los hay), usando un [bloque unificado de primitiva (PUB)](https://quantum.cloud.ibm.com/docs/es/guides/get-started-with-sampler).

In [ ]:
# Crea la instancia de Estimator.

estimator = Estimator(mode=backend)
estimator.options.resilience_level = 1
estimator.options.default_shots = 5000

mapped_observables = [
    observable.apply_layout(isa_circuit.layout) for observable in observables
]

# Un PUB, con un circuito que se ejecuta contra seis observables distintos.
job = estimator.run([(isa_circuit, mapped_observables)])

# Usa el Job ID para recuperar los datos del trabajo más tarde
print(f">>> Job ID: {job.job_id()}")

**Plan para la sesión en vivo:** si enviaste el trabajo antes (y anotaste su Job ID), recupéralo aquí en lugar de esperar la cola. Ejecuta esta celda **en lugar de** la anterior.

In [ ]:
# Recuperar un trabajo enviado antes de la sesión (pega el Job ID entre comillas)
# job = service.job("PEGA_AQUÍ_EL_JOB_ID")
# print("Estado:", job.status())

Después de enviar un trabajo, puedes esperar a que termine dentro de tu sesión actual de Python, o usar el `job_id` para recuperar los datos más tarde. (Consulta la [sección sobre cómo recuperar trabajos](https://quantum.cloud.ibm.com/docs/es/guides/monitor-job#retrieve-job-results-at-a-later-time) para más detalles).

Cuando el trabajo termine, examina su salida con el atributo `result()` del trabajo.

In [ ]:
# Este es el resultado de todo el envío. Enviaste un PUB,
# así que contiene un resultado interno (y algunos metadatos propios).
job_result = job.result()

# Este es el resultado de nuestro único PUB, que tenía seis observables,
# así que contiene información de los seis.
pub_result = job.result()[0]

<div class="alert alert-block alert-info">
<b>Alternativa: ejecutar el ejemplo con un simulador</b>

Cuando ejecutas tu programa cuántico en un dispositivo real, la carga de trabajo debe esperar en una cola antes de ejecutarse. Para ahorrar tiempo, puedes usar en su lugar el siguiente código para ejecutar esta pequeña carga de trabajo en el `fake_provider` con el modo de pruebas local de `qiskit-ibm-runtime`. Ten en cuenta que esto solo es posible con un circuito pequeño. Cuando escales en la siguiente sección, necesitarás un dispositivo real.

*Nota de la traducción:* el código de este recuadro está en la celda de abajo para que puedas ejecutarlo directamente. Ejecútalo **en lugar de** las tres celdas de hardware anteriores.
</div>

In [ ]:
# Usa el siguiente código si quieres usar un simulador:

from qiskit_ibm_runtime.fake_provider import FakeBelemV2
backend = FakeBelemV2()
estimator = Estimator(backend)

# Convierte a un circuito ISA y a observables mapeados al layout.

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
mapped_observables = [
    observable.apply_layout(isa_circuit.layout) for observable in observables
]

job = estimator.run([(isa_circuit, mapped_observables)])
result = job.result()

# Este es el resultado de todo el envío. Enviaste un PUB,
# así que contiene un resultado interno (y algunos metadatos propios).

job_result = job.result()

# Este es el resultado de nuestro único PUB, que tenía seis observables,
# así que contiene información de los seis.

pub_result = job.result()[0]

### Paso 4. Analizar los resultados

En el paso de análisis normalmente se post-procesan los resultados usando, por ejemplo, mitigación de errores de medición o extrapolación a ruido cero (ZNE). Puedes alimentar con estos resultados otro flujo de trabajo para seguir analizándolos, o preparar una gráfica de los valores y datos clave. En general, este paso depende de tu problema. En este ejemplo, se grafica cada uno de los valores esperados medidos para nuestro circuito.

Los valores esperados y las desviaciones estándar de los observables que especificaste para Estimator se obtienen con los atributos `PubResult.data.evs` y `PubResult.data.stds` del resultado del trabajo. Para obtener los resultados de Sampler, usa la función `PubResult.data.meas.get_counts()`, que devuelve un `dict` de mediciones con cadenas de bits como claves y sus conteos como valores. Para más información, consulta la guía [Inicio rápido de Sampler](https://quantum.cloud.ibm.com/docs/es/guides/get-started-with-sampler).

In [ ]:
# Grafica el resultado

values = pub_result.data.evs

errors = pub_result.data.stds

# gráfica
plt.plot(observables_labels, values, "-o")
plt.xlabel("Observables")
plt.ylabel("Valores")
plt.show()

Observa que, para los qubits 0 y 1, los valores esperados individuales de X y de Z son 0, mientras que las correlaciones (`XX` y `ZZ`) son 1. Esta es una señal característica del entrelazamiento cuántico.

## Escalar a un gran número de qubits

En computación cuántica, el trabajo a escala de utilidad es crucial para avanzar en el campo. Este tipo de trabajo requiere hacer cálculos a una escala mucho mayor, con circuitos que pueden usar más de 100 qubits y más de 1000 compuertas. Este ejemplo muestra cómo hacer trabajo a escala de utilidad en las QPU de IBM® creando y analizando un estado GHZ de 100 qubits. Usa el flujo de los patrones de Qiskit y termina midiendo el valor esperado $\langle Z_0 Z_i \rangle$ para cada qubit.

### Paso 1. Mapear el problema

Escribe una función que devuelva un `QuantumCircuit` que prepare un estado GHZ de $n$ qubits (esencialmente, un estado de Bell extendido). Luego usa esa función para preparar un estado GHZ de 100 qubits y reunir los observables que se van a medir.

In [ ]:
def get_qc_for_n_qubit_GHZ_state(n: int) -> QuantumCircuit:
    """Esta función crea un qiskit.QuantumCircuit (qc)
        para un estado GHZ de n qubits.

    Args:
        n (int): Número de qubits del estado GHZ de n qubits

    Returns:
        QuantumCircuit: Circuito cuántico que genera el estado GHZ de n qubits,
            suponiendo que todos los qubits empiezan en el estado 0
    """
    if isinstance(n, int) and n >= 2:
        qc = QuantumCircuit(n)
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
    else:
        raise Exception("n no es una entrada válida")
    return qc


# Crea un circuito nuevo con 100 qubits en el estado GHZ
n = 100
qc = get_qc_for_n_qubit_GHZ_state(n)

A continuación, se mapean los operadores de interés. Este ejemplo usa los operadores `ZZ` entre qubits para examinar el comportamiento a medida que se alejan entre sí. Valores esperados cada vez menos precisos (corrompidos) entre qubits distantes revelarían el nivel de ruido presente.

In [ ]:
# ZZII...II, ZIZI...II, ... , ZIII...IZ
operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (n - 2 - i) for i in range(n - 1)
]

operators = [SparsePauliOp(operator) for operator in operator_strings]

### Paso 2. Optimizar el problema para ejecutarlo en hardware cuántico

El siguiente código transforma el circuito y los observables para que se ajusten a la ISA del dispositivo. Requiere que ya hayas [guardado tus credenciales](https://quantum.cloud.ibm.com/docs/es/guides/cloud-setup).

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(
    simulator=False, operational=True, min_num_qubits=100
)
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

isa_circuit = pm.run(qc)
isa_operators_list = [op.apply_layout(isa_circuit.layout) for op in operators]

<div class="alert alert-block alert-info">
<b>Nota de la traducción: alternativa con simulador</b>

El tutorial original indica que esta parte necesita un dispositivo real: un simulador local no puede con 100 qubits. Para seguir el flujo sin cuenta, la celda de abajo repite los pasos 1 y 2 con **5 qubits** en `FakeBelemV2`. Ejecútala **en lugar de** la celda anterior; las celdas siguientes funcionan igual con ambas opciones.
</div>

In [ ]:
# Alternativa con simulador: ejecuta esta celda en lugar de la anterior.
from qiskit_ibm_runtime.fake_provider import FakeBelemV2

n = 5
qc = get_qc_for_n_qubit_GHZ_state(n)
operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (n - 2 - i) for i in range(n - 1)
]
operators = [SparsePauliOp(operator) for operator in operator_strings]

backend = FakeBelemV2()
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

isa_circuit = pm.run(qc)
isa_operators_list = [op.apply_layout(isa_circuit.layout) for op in operators]

### Paso 3. Ejecutar en hardware

Envía el trabajo y activa la supresión de errores con una técnica para reducirlos llamada [desacoplamiento dinámico](https://quantum.cloud.ibm.com/docs/es/api/qiskit-ibm-runtime/options-models-dynamical-decoupling-options) (*dynamical decoupling*). El nivel de resiliencia (*resilience level*) indica cuánta resistencia a los errores se incorpora. Los niveles más altos dan resultados más precisos, a costa de tiempos de procesamiento más largos. Para más explicación sobre las opciones que se configuran en el siguiente código, consulta [Técnicas de mitigación y supresión de errores](https://quantum.cloud.ibm.com/docs/es/guides/error-mitigation-and-suppression-techniques).

In [ ]:
options = EstimatorOptions()
options.resilience_level = 1
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"

# Crea un objeto Estimator
estimator = Estimator(backend, options=options)

In [ ]:
# Envía el circuito a Estimator
job = estimator.run([(isa_circuit, isa_operators_list)])
job_id = job.job_id()
print(job_id)

### Paso 4. Post-procesar los resultados

Cuando el trabajo termine, grafica los resultados y observa que $\langle Z_0 Z_i \rangle$ disminuye a medida que aumenta $i$, aunque en una simulación ideal todos los $\langle Z_0 Z_i \rangle$ deberían valer 1.

In [ ]:
# datos
data = list(range(1, len(operators) + 1))  # Distancia entre los operadores Z
result = job.result()[0]
values = result.data.evs  # Valor esperado de cada operador Z.
values = [
    v / values[0] for v in values
]  # Normaliza los valores esperados para evaluar cómo decaen con la distancia.

# gráfica
plt.plot(data, values, marker="o", label=f"Estado GHZ de {n} qubits")
plt.xlabel("Distancia entre qubits $i$")
plt.ylabel(r"$\langle Z_i Z_0 \rangle / \langle Z_1 Z_0 \rangle $")
plt.legend()
plt.show()

La gráfica anterior muestra que, a medida que aumenta la distancia entre los qubits, la señal decae por la presencia de ruido.

## Siguientes pasos

<div class="alert alert-block alert-success">
<b>Recomendaciones</b>

* Prueba uno de estos tutoriales:
  * [Estimación de la energía del estado base de la cadena de Heisenberg con VQE](https://quantum.cloud.ibm.com/docs/es/tutorials/spin-chain-vqe)
  * Resuelve problemas de optimización con [QAOA](https://quantum.cloud.ibm.com/docs/es/tutorials/quantum-approximate-optimization-algorithm)
  * Entrena modelos de [kernel cuántico](https://quantum.cloud.ibm.com/docs/es/tutorials/quantum-kernel-training) para tareas de aprendizaje automático
* Encuentra instrucciones detalladas de instalación en la guía [Instalar Qiskit](https://quantum.cloud.ibm.com/docs/es/guides/install-qiskit).
* Si prefieres no instalar Qiskit localmente, consulta las opciones para usar Qiskit en un [entorno de desarrollo en línea](https://quantum.cloud.ibm.com/docs/es/guides/online-lab-environments).
* Para guardar las credenciales de varias cuentas o especificar otras opciones de cuenta, consulta las instrucciones detalladas en la guía [Guardar tus credenciales de acceso](https://quantum.cloud.ibm.com/docs/es/guides/save-credentials#save-your-access-credentials).
</div>

© IBM Corp., 2017-2026. Traducción al español para el Qiskit Fall Fest 2026.